# 🧬 SimDock Polymer v2.0 - Cloud GPU Environment
Run this notebook step-by-step to execute the *real* physics simulation (GNINA docking + OpenMM 10ns MD) on a free Colab GPU.

**Important:** Run each cell in order. Cell 1 will restart the kernel automatically — that's normal!

In [ ]:
# 1. Setup Conda (The kernel will automatically restart after this finishes. That is normal!)
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# 2. Install heavy chemistry dependencies via Conda
import condacolab
condacolab.check()
print('✅ Conda environment is active.')
!conda install -y -c conda-forge openmm mdtraj rdkit biopython openmmforcefields openff-toolkit ambertools
print('\n✅ Conda packages installed.')

In [ ]:
# 3. Install pip packages INTO the conda Python (not the system one!)
import sys
print(f'Using Python: {sys.executable}')
!{sys.executable} -m pip install --quiet meeko gemmi pdbfixer fastapi uvicorn pydantic matplotlib pyyaml pandas scipy numpy

# Download docking engines & Cloudflare tunnel binary
!wget -q https://github.com/gnina/gnina/releases/download/v1.0.3/gnina -O /usr/bin/gnina
!chmod +x /usr/bin/gnina
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O /usr/bin/vina
!chmod +x /usr/bin/vina
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/bin/cloudflared
!chmod +x /usr/bin/cloudflared
print('\n✅ Pip packages and system binaries installed.')

In [ ]:
# 4. Verify OpenMM + GPU environment BEFORE launching the server
import sys, importlib
print(f'Python executable: {sys.executable}')
print(f'Python version:    {sys.version}')
print()

# Test critical imports one by one
packages = ['openmm', 'mdtraj', 'rdkit', 'openmmforcefields', 'openff.toolkit']
all_ok = True
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', getattr(mod, 'version', '?'))
        print(f'  ✅ {pkg:25s} → {ver}')
    except ImportError as e:
        print(f'  ❌ {pkg:25s} → MISSING: {e}')
        all_ok = False

# Check GPU platform
print()
try:
    import openmm
    for i in range(openmm.Platform.getNumPlatforms()):
        p = openmm.Platform.getPlatform(i)
        print(f'  Platform {i}: {p.getName()}')
    # Try to use CUDA
    try:
        cuda = openmm.Platform.getPlatformByName('CUDA')
        print(f'\n  🚀 CUDA platform available! MD will run on GPU.')
    except Exception:
        print(f'\n  ⚠️ CUDA not available, will fall back to CPU (slower but still real MD).')
except Exception as e:
    print(f'  ❌ OpenMM platform check failed: {e}')

if all_ok:
    print('\n🎉 All packages verified — REAL MD simulations will run!')
else:
    print('\n⚠️ Some packages are missing. Fix the errors above before launching.')

In [ ]:
# 5. Retrieve project source files
# Clones from your GitHub repository directly. If cloning fails (e.g. private repo restrictions),
# it falls back to unzipping 'SimDock_Project.zip' if you uploaded it manually.
import os
if not os.path.exists('polymerDock'):
    !git clone https://github.com/messiay/PolymerDock.git polymerDock || (unzip -q -o SimDock_Project.zip -d polymerDock && echo 'Fallback to local SimDock_Project.zip successful')
else:
    print('polymerDock directory already exists, skipping clone.')
%cd polymerDock

In [ ]:
# 6. Launch the SimDock Web Application with Cloudflare Tunnel!
import sys
import subprocess
import time
import re
import os

# Build the environment for the subprocess — inherit the FULL conda env
env = os.environ.copy()
# Ensure the conda bin dir is first on PATH so openmm/rdkit resolve correctly
conda_prefix = os.environ.get('CONDA_PREFIX', '')
if conda_prefix:
    conda_bin = os.path.join(conda_prefix, 'bin')
    env['PATH'] = conda_bin + ':' + env.get('PATH', '')
    conda_lib = os.path.join(conda_prefix, 'lib')
    env['LD_LIBRARY_PATH'] = conda_lib + ':' + env.get('LD_LIBRARY_PATH', '')

print(f'Python executable: {sys.executable}')
print(f'CONDA_PREFIX:      {conda_prefix}')
print()

# Quick sanity check — can THIS Python import openmm?
try:
    import openmm
    print(f'✅ OpenMM {openmm.__version__} is importable. Real MD will run.')
except ImportError as e:
    print(f'❌ OpenMM import failed: {e}')
    print('   The simulation will fall back to MOCK data. Check cells 2-4.')
print()

print('Starting FastAPI Uvicorn Server...')
server_log = open('server.log', 'w')
proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api:app', '--host', '0.0.0.0', '--port', '8501'],
    env=env,
    stdout=server_log,
    stderr=subprocess.STDOUT
)
time.sleep(4)

# Check if server started successfully
if proc.poll() is not None:
    print('❌ Server crashed on startup! Check server.log:')
    with open('server.log') as f:
        print(f.read())
else:
    print('✅ Server is running on port 8501.')

print('\nStarting Cloudflare Tunnel...')
cf_log_path = 'cloudflare.log'
with open(cf_log_path, 'w') as f:
    subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8501'], stdout=f, stderr=f)

# Poll the log file for the public tunnel URL
tunnel_url = None
print('Retrieving public HTTPS URL from Cloudflare...')
for _ in range(15):
    time.sleep(1)
    if os.path.exists(cf_log_path):
        with open(cf_log_path, 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print('\n✅ SUCCESS! Click the link below to open your app in a new tab:')
    print(f'\n👉 {tunnel_url} 👈\n')
else:
    print('\n⚠️ Could not automatically detect Cloudflare URL.')
    print('Check cloudflare.log for details, or try the port window fallback:')
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(8501)
    except Exception:
        pass

In [ ]:
# 7. (Debug) View server logs if something goes wrong
# Run this cell if the dashboard says 'OpenMM unavailable' or the server crashed.
with open('server.log') as f:
    print(f.read())